In [86]:
import pandas as pd

In [87]:
# ---------- 1) Download Census ACS 5-year county data (2023) ----------
# To pull socioeconomic variables from US Census API for county-level analysis
poverty_url = "https://api.census.gov/data/2023/acs/acs5/subject?get=NAME,S1701_C03_001E&for=county:*" # Poverty status in the past 12 months below poverty level
income_url  = "https://api.census.gov/data/2023/acs/acs5/subject?get=NAME,S1901_C01_012E&for=county:*" # Median household income in the past 12 months (in 2023 inflation-adjusted dollars)
edu_url     = "https://api.census.gov/data/2023/acs/acs5/subject?get=NAME,S1501_C02_015E&for=county:*" # Educational attainment for the population 25 years and over: Bachelor's degree or higher

poverty = pd.read_json(poverty_url) 
income  = pd.read_json(income_url)
edu     = pd.read_json(edu_url)

In [ ]:
# # Inspect raw Census API output: Census API returns the first row as column labels.
#   We verify the structure before cleaning.
print("Raw poverty head:")
print(poverty.head())

Raw poverty head:
                         0               1      2       3
0                     NAME  S1701_C03_001E  state  county
1  Autauga County, Alabama            10.7     01     001
2  Baldwin County, Alabama            10.5     01     003
3  Barbour County, Alabama            21.9     01     005
4     Bibb County, Alabama            20.5     01     007


In [ ]:
# ---------- 2) Fix headers ----------
# Convert the first row into column headers so we can work with clean column names.
poverty.columns = poverty.iloc[0] # Set the first row as column headers
poverty = poverty[1:] # Remove the first row which is now the header
print(poverty.head(5)) # Verify the first few rows to confirm headers are set correctly

#do the same for income and edu
income.columns = income.iloc[0]
income = income[1:]

edu.columns = edu.iloc[0]
edu = edu[1:]

0                     NAME S1701_C03_001E state county
1  Autauga County, Alabama           10.7    01    001
2  Baldwin County, Alabama           10.5    01    003
3  Barbour County, Alabama           21.9    01    005
4     Bibb County, Alabama           20.5    01    007
5   Blount County, Alabama           14.1    01    009


In [90]:
# ---------- 3) Create county FIPS ----------
#   County FIPS = state (2 digits) + county (3 digits).
#   This will be the merge key for county-level fusion.
for df in [poverty, income, edu]:
    df["FIPS"] = df["state"] + df["county"]

print(poverty.head(5)) # Check the new FIPS column
print(income.head(5))
print(edu.head(5))

0                     NAME S1701_C03_001E state county   FIPS
1  Autauga County, Alabama           10.7    01    001  01001
2  Baldwin County, Alabama           10.5    01    003  01003
3  Barbour County, Alabama           21.9    01    005  01005
4     Bibb County, Alabama           20.5    01    007  01007
5   Blount County, Alabama           14.1    01    009  01009
0                     NAME S1901_C01_012E state county   FIPS
1  Autauga County, Alabama          69841    01    001  01001
2  Baldwin County, Alabama          75019    01    003  01003
3  Barbour County, Alabama          44290    01    005  01005
4     Bibb County, Alabama          51215    01    007  01007
5   Blount County, Alabama          61096    01    009  01009
0                     NAME S1501_C02_015E state county   FIPS
1  Autauga County, Alabama           28.3    01    001  01001
2  Baldwin County, Alabama           32.8    01    003  01003
3  Barbour County, Alabama           11.5    01    005  01005
4     Bi

In [ ]:
# ---------- 4) Rename variables ----------
# Replace Census cryptic variable codes with meaningful names for readability and analysis.
poverty = poverty.rename(columns={"S1701_C03_001E": "poverty_rate"})
income  = income.rename(columns={"S1901_C01_012E": "median_income"})
edu     = edu.rename(columns={"S1501_C02_015E": "bachelor_plus_rate"})
print(poverty.head(5)) # Check renamed columns
print(income.head(5))
print(edu.head(5))

0                     NAME poverty_rate state county   FIPS
1  Autauga County, Alabama         10.7    01    001  01001
2  Baldwin County, Alabama         10.5    01    003  01003
3  Barbour County, Alabama         21.9    01    005  01005
4     Bibb County, Alabama         20.5    01    007  01007
5   Blount County, Alabama         14.1    01    009  01009
0                     NAME median_income state county   FIPS
1  Autauga County, Alabama         69841    01    001  01001
2  Baldwin County, Alabama         75019    01    003  01003
3  Barbour County, Alabama         44290    01    005  01005
4     Bibb County, Alabama         51215    01    007  01007
5   Blount County, Alabama         61096    01    009  01009
0                     NAME bachelor_plus_rate state county   FIPS
1  Autauga County, Alabama               28.3    01    001  01001
2  Baldwin County, Alabama               32.8    01    003  01003
3  Barbour County, Alabama               11.5    01    005  01005
4     Bibb

In [92]:
# ---------- 5) Merge Census variables into one dataset ----------
# Combine poverty, income, and education variables into a single county-level dataframe.
census_df = poverty[["FIPS", "NAME", "poverty_rate"]].merge(
    income[["FIPS", "median_income"]], on="FIPS").merge(
        edu[["FIPS", "bachelor_plus_rate"]], on="FIPS")
print(census_df.head(5))

0   FIPS                     NAME poverty_rate median_income  \
0  01001  Autauga County, Alabama         10.7         69841   
1  01003  Baldwin County, Alabama         10.5         75019   
2  01005  Barbour County, Alabama         21.9         44290   
3  01007     Bibb County, Alabama         20.5         51215   
4  01009   Blount County, Alabama         14.1         61096   

0 bachelor_plus_rate  
0               28.3  
1               32.8  
2               11.5  
3               11.5  
4               15.6  


In [93]:
# ---------- 6) Convert to numeric ----------
# ensure variables are usable for statistical analysis
# check data types
print(census_df.dtypes)

census_df["poverty_rate"] = pd.to_numeric(census_df["poverty_rate"], errors="coerce") # Convert poverty_rate to numeric, coercing errors to NaN
census_df["median_income"] = pd.to_numeric(census_df["median_income"], errors="coerce")
census_df["bachelor_plus_rate"] = pd.to_numeric(census_df["bachelor_plus_rate"], errors="coerce")
print(census_df.dtypes) # Check that the columns are now numeric

0
FIPS                  object
NAME                  object
poverty_rate          object
median_income         object
bachelor_plus_rate    object
dtype: object
0
FIPS                   object
NAME                   object
poverty_rate          float64
median_income           int64
bachelor_plus_rate    float64
dtype: object


In [94]:
# ---------- 7) Validate Census dataset ----------
# Confirm dataset shape and check missing values.
print("\nCensus dataset shape:", census_df.shape)
print(census_df.head())
print("\nMissing values check:")
print(census_df.isna().sum())


Census dataset shape: (3222, 5)
0   FIPS                     NAME  poverty_rate  median_income  \
0  01001  Autauga County, Alabama          10.7          69841   
1  01003  Baldwin County, Alabama          10.5          75019   
2  01005  Barbour County, Alabama          21.9          44290   
3  01007     Bibb County, Alabama          20.5          51215   
4  01009   Blount County, Alabama          14.1          61096   

0  bachelor_plus_rate  
0                28.3  
1                32.8  
2                11.5  
3                11.5  
4                15.6  

Missing values check:
0
FIPS                  0
NAME                  0
poverty_rate          0
median_income         0
bachelor_plus_rate    0
dtype: int64


In [ ]:
# ---------- 8) Load NCES 2022–2023 district-level datasets ----------
# We need district-level outcomes such as dropout and graduation counts.
# NCES CCD provides district (LEA) datasets.
lea_path = "nces_2022_2023/ccd_lea_029_2223_w_1a_083023.csv"
dropout_path = "nces_2022_2023/CCD_LEA_032_2223_L_1a_PUB_050824.CSV"
grad_path = "nces_2022_2023/CCD_LEA_040_2223_L_1a_PUB_050824.CSV"

lea_df = pd.read_csv(lea_path, low_memory=False) # Read the LEA file, setting low_memory=False to avoid dtype warnings
dropout_df = pd.read_csv(dropout_path, low_memory=False) # Read the dropout file, setting low_memory=False to avoid dtype warnings
grad_df = pd.read_csv(grad_path, low_memory=False) # Read the graduate file, setting low_memory=False to avoid dtype warnings

(19714, 58) (206074, 15) (344828, 15)


In [97]:
# Validate NCES dataset sizes 
# Check whether dataset sizes match expectation.
# LEA Universe file should be ~ 10k-20k rows (districts).
# Dropout and grad files may be larger due to subgroup breakdowns.
print("\nLEA dataset shape:", lea_df.shape)
print("Dropout dataset shape:", dropout_df.shape)
print("Graduate dataset shape:", grad_df.shape)


LEA dataset shape: (19714, 58)
Dropout dataset shape: (206074, 15)
Graduate dataset shape: (344828, 15)


In [98]:
# ---------- 9) Inspect column names----------
# Confirm that LEAID exists in all datasets as the district identifier.
print("LEA columns:\n", lea_df.columns.tolist())
print("Dropout columns:\n", dropout_df.columns.tolist())
print("Graduate columns:\n", grad_df.columns.tolist())

LEA columns:
 ['SCHOOL_YEAR', 'FIPST', 'STATENAME', 'ST', 'LEA_NAME', 'STATE_AGENCY_NO', 'UNION', 'ST_LEAID', 'LEAID', 'MSTREET1', 'MSTREET2', 'MSTREET3', 'MCITY', 'MSTATE', 'MZIP', 'MZIP4', 'LSTREET1', 'LSTREET2', 'LSTREET3', 'LCITY', 'LSTATE', 'LZIP', 'LZIP4', 'PHONE', 'WEBSITE', 'SY_STATUS', 'SY_STATUS_TEXT', 'UPDATED_STATUS', 'UPDATED_STATUS_TEXT', 'EFFECTIVE_DATE', 'LEA_TYPE', 'LEA_TYPE_TEXT', 'OUT_OF_STATE_FLAG', 'CHARTER_LEA', 'CHARTER_LEA_TEXT', 'NOGRADES', 'G_PK_OFFERED', 'G_KG_OFFERED', 'G_1_OFFERED', 'G_2_OFFERED', 'G_3_OFFERED', 'G_4_OFFERED', 'G_5_OFFERED', 'G_6_OFFERED', 'G_7_OFFERED', 'G_8_OFFERED', 'G_9_OFFERED', 'G_10_OFFERED', 'G_11_OFFERED', 'G_12_OFFERED', 'G_13_OFFERED', 'G_UG_OFFERED', 'G_AE_OFFERED', 'GSLO', 'GSHI', 'LEVEL', 'IGOFFERED', 'OPERATIONAL_SCHOOLS']
Dropout columns:
 ['SCHOOL_YEAR', 'FIPST', 'STATENAME', 'ST', 'LEA_NAME', 'STATE_AGENCY_NO', 'UNION', 'ST_LEAID', 'LEAID', 'GRADE', 'RACE_ETHNICITY', 'SEX', 'STUDENT_COUNT', 'TOTAL_INDICATOR', 'DMS_FLAG']
G

In [99]:
# ---------- 10): Confirm common merge keys ----------
#   Identify shared columns and confirm LEAID exists as a unique district identifier.
common_cols = set(lea_df.columns).intersection(set(dropout_df.columns)).intersection(set(grad_df.columns))
print("\nCommon columns in all 3 datasets:", common_cols)


Common columns in all 3 datasets: {'ST', 'UNION', 'FIPST', 'ST_LEAID', 'LEAID', 'LEA_NAME', 'SCHOOL_YEAR', 'STATE_AGENCY_NO', 'STATENAME'}


In [ ]:
# ---------- 11):Attempt a naive merge (for diagnostic purposes) ----------

naive_merge = lea_df.merge(dropout_df, on="LEAID", how="left")
naive_merge = naive_merge.merge(grad_df, on="LEAID", how="left")

print("\nNaive merged shape (should NOT be this large):", naive_merge.shape)


Naive merged shape (should NOT be this large): (5156255, 86)


In [ ]:
# ---------- 11) Merge datasets (This step turns out explosion problem) ----------
# This step is used to demonstrate the join explosion problem when mergin datasets and will not be used in the final pipeline.

key = "LEAID" if "LEAID" in lea_df.columns else None # Check if LEAID is in the columns, otherwise set key to None
if key is None:
    raise ValueError("LEAID not found. Need to inspect column names.") # If LEAID is not found, raise an error to inspect column names

# Merge dropout + grad into LEA base
nces_df = lea_df.merge(dropout_df, on=key, how="left", suffixes=("", "_drop")) # Merge dropout data into LEA data, using left join to keep all LEAs, and adding suffixes to distinguish columns
nces_df = nces_df.merge(grad_df, on=key, how="left", suffixes=("", "_grad"))

print("\nMerged NCES shape:", nces_df.shape)
print(nces_df[[key]].head())


Merged NCES shape: (5156255, 86)
    LEAID
0  100002
1  100002
2  100002
3  100002
4  100002


In [102]:
# ---------- 12) : Verify subgroup breakdowns ----------
#   Confirm dropout and grad datasets contain multiple demographic categories per LEAID.
print("\nDropout TOTAL_INDICATOR distribution:")
print(dropout_df["TOTAL_INDICATOR"].value_counts().head(10))

print("\nGraduate TOTAL_INDICATOR distribution:")
print(grad_df["TOTAL_INDICATOR"].value_counts().head(10))

print("\nDropout breakdown examples:")
print(dropout_df[["GRADE", "RACE_ETHNICITY", "SEX"]].drop_duplicates().head(10))

print("\nGraduate breakdown examples:")
print(grad_df[["CREDENTIAL", "RACE_ETHNICITY", "SEX"]].drop_duplicates().head(10))


Dropout TOTAL_INDICATOR distribution:
TOTAL_INDICATOR
Category Set A - By Race/Ethnicity; Sex; Grade    192136
Education Unit Total                               13938
Name: count, dtype: int64

Graduate TOTAL_INDICATOR distribution:
TOTAL_INDICATOR
Category Set A - By Race/Ethnicity; Sex; Credential    309050
Subtotal 1 - By Credential                              22075
Education Unit Total                                    13703
Name: count, dtype: int64

Dropout breakdown examples:
         GRADE                             RACE_ETHNICITY     SEX
0  Grades 9-12           American Indian or Alaska Native  Female
1  Grades 9-12           American Indian or Alaska Native    Male
2  Grades 9-12                                      Asian  Female
3  Grades 9-12                                      Asian    Male
4  Grades 9-12                  Black or African American  Female
5  Grades 9-12                  Black or African American    Male
6  Grades 9-12                            Hisp

In [ ]:
# Conclusion:
#   The dropout and grad datasets are NOT one-row-per-district.
#   They contain subgroup-level rows (race, sex, grade, credential).
#   Therefore, we must aggregate them before merging with lea_df.

In [103]:
# ---------- 13）: Extract district total dropout counts ----------
#   "Education Unit Total" indicates the district-level total (all subgroups combined).
#   This avoids double counting and provides a clean district-level measure.

dropout_total = dropout_df[
    dropout_df["TOTAL_INDICATOR"] == "Education Unit Total"
].copy()

print("\nDropout total rows shape:", dropout_total.shape)


Dropout total rows shape: (13938, 15)


In [104]:
# Aggregate dropout count per district (LEAID)
#   Even within total rows, some LEAIDs may appear multiple times (rare), so we sum as a safe method.
dropout_agg = dropout_total.groupby("LEAID")["STUDENT_COUNT"].sum().reset_index()
dropout_agg = dropout_agg.rename(columns={"STUDENT_COUNT": "dropout_count"})

print("Dropout aggregated shape:", dropout_agg.shape)
print(dropout_agg.head())

Dropout aggregated shape: (13938, 2)
    LEAID  dropout_count
0  100002            0.0
1  100005           32.0
2  100006           15.0
3  100007           30.0
4  100008           31.0


In [105]:
# ---------- 14）: Extract district total graduate counts ----------
#   We also filter for "Education Unit Total" to ensure district-level totals.

grad_total = grad_df[
    grad_df["TOTAL_INDICATOR"] == "Education Unit Total"
].copy()

print("\nGraduate total rows shape:", grad_total.shape)


Graduate total rows shape: (13703, 15)


In [108]:
# Aggregate graduate counts by district and credential type
#   This allows us to keep separate counts for diploma types if available.
grad_agg = grad_total.groupby(["LEAID", "CREDENTIAL"])["STUDENT_COUNT"].sum().reset_index()

print("Graduate aggregated shape:", grad_agg.shape)
print(grad_agg.head())

Graduate aggregated shape: (13703, 3)
    LEAID         CREDENTIAL  STUDENT_COUNT
0  100002  No Category Codes            0.0
1  100005  No Category Codes          376.0
2  100006  No Category Codes          401.0
3  100007  No Category Codes          986.0
4  100008  No Category Codes          993.0


In [110]:
# Pivot to wide format: each credential becomes a column
#   This makes the dataset easier to merge and use in ML models.
grad_pivot = grad_agg.pivot(
    index="LEAID",
    columns="CREDENTIAL",
    values="STUDENT_COUNT"
).reset_index()

print(grad_pivot.head())


CREDENTIAL   LEAID  No Category Codes
0           100002                0.0
1           100005              376.0
2           100006              401.0
3           100007              986.0
4           100008              993.0


In [111]:
# Clean column names for modeling stability
#   Replace spaces with underscores and strip whitespace.
grad_pivot.columns = [str(c).strip().replace(" ", "_") for c in grad_pivot.columns]

print("Graduation pivot shape:", grad_pivot.shape)
print(grad_pivot.head())

Graduation pivot shape: (13703, 2)
    LEAID  No_Category_Codes
0  100002                0.0
1  100005              376.0
2  100006              401.0
3  100007              986.0
4  100008              993.0


In [112]:
# ---------- 15): Merge aggregated dropout + graduation into district LEA table ----------
#   lea_df provides district metadata and identifiers.
#   dropout_agg and grad_pivot provide outcome variables.

nces_clean = lea_df.merge(dropout_agg, on="LEAID", how="left")
nces_clean = nces_clean.merge(grad_pivot, on="LEAID", how="left")

print("\nFinal cleaned NCES dataset shape:", nces_clean.shape)
print(nces_clean[["LEAID", "LEA_NAME", "FIPST", "dropout_count"]].head())


Final cleaned NCES dataset shape: (19714, 60)
    LEAID                LEA_NAME  FIPST  dropout_count
0  100002  Alabama Youth Services      1            0.0
1  100005        Albertville City      1           32.0
2  100006         Marshall County      1           15.0
3  100007             Hoover City      1           30.0
4  100008            Madison City      1           31.0


In [114]:
# ---------- 16): Check if county code exists in NCES ----------
#   Determine whether we can directly fuse NCES with Census at the county level.

possible_county_cols = [c for c in lea_df.columns if "CON" in c.upper() or "COUNT" in c.upper()]
print("\nPossible county columns in lea_df:", possible_county_cols)


Possible county columns in lea_df: []


In [ ]:
# ---------- 16): Merge aggregated dropout + graduation into district LEA table ----------